In [13]:
%load_ext autoreload
%autoreload 2
%load_ext line_profiler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


In [21]:
from tqdm import tqdm

import os
import torch
import numpy as np
from torch_geometric.data import Batch, HeteroData
from numpy.linalg import LinAlgError

from utils.evaluation import solve_sdp_cvxpy, solve_sdp_scs, solve_sdp_scs_relaxed
from torch_geometric.utils import to_dense_adj

In [15]:
rng = np.random.RandomState(1)

In [16]:
root = 'datasets/maxcut_reg_30_5'
os.mkdir(root)
os.mkdir(os.path.join(root, 'processed'))

FileExistsError: [Errno 17] File exists: 'datasets/maxcut_reg_30_5'

In [22]:
import networkx as nx
from torch_geometric.utils.convert import from_networkx


def erdos_renyi_generator(rng, n_min=100, n_max=100, p_min=0.15, p_max=0.15):
    n = rng.randint(n_min, n_max + 1)
    p = rng.uniform(p_min, p_max)
    G = nx.erdos_renyi_graph(n, p, rng)
    return from_networkx(G)


def barabasi_albert_generator(rng, n_min=100, n_max=100, m_min=4, m_max=4):
    n = rng.randint(n_min, n_max + 1)
    m = rng.randint(n_min, n_max + 1)
    G = nx.barabasi_albert_graph(n, m, rng)
    return from_networkx(G)


def regular_generator(rng, n_min=100, n_max=100, d_min=3, d_max=3):
    n = rng.randint(n_min, n_max + 1)
    d = rng.randint(d_min, d_max + 1)
    G = nx.random_regular_graph(d, n, rng)
    return from_networkx(G)

### Max cut - Erdos

In [ ]:
def generate_max_cut_sdp(nnodes, density):
    data = erdos_renyi_generator(rng, nnodes, nnodes, density, density)
    N = nnodes
    edge_index = data.edge_index
    E = edge_index.shape[1]
    adj = to_dense_adj(edge_index, max_num_nodes=N)[0].numpy()

    A = []
    b = []
    # diagonals being 1
    for i in range(N):
        const = np.zeros((N, N))
        const[i, i] = 1
        A.append(const)
        b.append(1)

    return adj, np.stack(A, axis=-1).astype(np.float32), np.array(b, dtype=np.float32)

### Max cut - regular

In [23]:
def generate_max_cut_regular_sdp(nnodes, deg):
    data = regular_generator(rng, nnodes, nnodes, deg, deg)
    N = nnodes
    edge_index = data.edge_index
    E = edge_index.shape[1]
    adj = to_dense_adj(edge_index, max_num_nodes=N)[0].numpy()

    A = []
    b = []
    # diagonals being 1
    for i in range(N):
        const = np.zeros((N, N))
        const[i, i] = 1
        A.append(const)
        b.append(1)

    return adj, np.stack(A, axis=-1).astype(np.float32), np.array(b, dtype=np.float32)

In [ ]:
from cvxpy import DCPError, DGPError, DPPError, SolverError
from utils.evaluation import map_vec, mat

In [ ]:
graphs = []
pkg_idx = 0
success_cnt = 0

max_iter = 12000
num = 10000

pbar = tqdm(range(max_iter))
for i in pbar:
    try:
        C, A, b = generate_lovasz_theta_er_sdp(100, 0.1)
        # sol, X, stat, times = solve_sdp_cvxpy(C, A, b, fnorm_strength, solver)
        X, y, dual, sol = solve_sdp_scs(C, A, b, 1.e-5)
        # assert stat == 'optimal'
        assert sol['info']['status'] == 'solved'
    except (LinAlgError, DCPError, DGPError, DPPError, SolverError, AssertionError):
        continue

    else:
        m = b.shape[0]
        n = C.shape[0]
        A = torch.from_numpy(A).float()
        A = A.reshape(-1, A.shape[-1]).T  # m, n**2
        A_where = torch.where(A)
        
        c2v_idx = torch.vstack(A_where)
        c2v_value = A[A_where][:, None]
        
        C = torch.from_numpy(C).float().reshape(-1)[None]
        # sparse vals obj connections
        C_where = torch.where(C)
        o2v_idx = torch.vstack(C_where)
        o2v_value = C[C_where][:, None]

        x = torch.from_numpy(X).float().reshape(-1)
        y = torch.from_numpy(y).float()
        dual = torch.from_numpy(dual).float().reshape(-1)

        data = HeteroData(
            cons={
                'num_nodes': m,
                'x': torch.empty(m, 0),
                 },
            vals={
                'num_nodes': n ** 2,
                'x': torch.empty(n ** 2, 0),
            },
            obj={
                'num_nodes': 1,
                'x': torch.ones(1).float(),
                 },
            cons__to__vals={'edge_index': c2v_idx,
                            'edge_attr': c2v_value},
            obj__to__vals={'edge_index': o2v_idx,
                            'edge_attr': o2v_value},
            x_solution=x,
            y_solution=y,
            dual_solution=dual,
            obj_solution=torch.tensor([sol['info']['pobj']]),
            b=torch.from_numpy(b).float(),
        )
        success_cnt += 1
        graphs.append(data)

    if len(graphs) >= 1000 or success_cnt == num:
        torch.save(Batch.from_data_list(graphs), f'{root}/processed/batch{pkg_idx}.pt')
        pkg_idx += 1
        graphs = []

    if success_cnt >= num:
        break

    pbar.set_postfix({'suc': success_cnt})

## save as test only

In [ ]:
from torch_geometric.data import InMemoryDataset

datas = torch.load(f'{root}/processed/batch0.pt')
datas = Batch.to_data_list(datas)
torch.save(InMemoryDataset().collate(datas), f'{root}/processed/test.pt')
torch.save(None, f'{root}/processed/train.pt')
torch.save(None, f'{root}/processed/valid.pt')

## save as normal dataset

In [17]:
from data.dataset import LPDataset

ds = LPDataset(root, 'valid')
assert not torch.isnan(ds.data.obj_solution).any()

/var/folders/6v/m172f7bs02l0tkwy4_9yxttm0000gn/T/ipykernel_5402/4038735500.py:4: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  assert not torch.isnan(ds.data.obj_solution).any()


In [20]:
ds[0].dual_solution.reshape(30, 30)

tensor([[ 4.0352e+00,  1.0587e-05,  9.1775e-06,  3.0952e-06, -5.9089e-06,
          9.9999e-01, -3.7209e-06,  9.6533e-06,  9.9999e-01, -4.6244e-06,
         -6.6098e-06,  4.2775e-06,  8.7602e-06, -1.0567e-05, -3.1781e-06,
         -9.3615e-06,  1.1771e-05,  1.2127e-06, -1.0131e-05,  1.0000e+00,
          2.5985e-07,  5.7683e-06,  1.1283e-05, -6.4674e-06,  2.4039e-06,
          1.0139e-05,  9.9999e-01, -3.2324e-06, -5.8148e-06,  1.0000e+00],
        [ 1.0587e-05,  3.6336e+00, -7.8198e-07, -6.8319e-06,  9.9999e-01,
         -1.1067e-05,  9.9999e-01,  1.6164e-05, -7.9714e-06,  2.1665e-06,
         -1.7666e-05,  1.5496e-05,  1.3098e-05,  9.9998e-01,  8.6137e-06,
          9.9998e-01,  1.8742e-05,  9.0644e-06, -1.2877e-05, -6.2992e-06,
          3.9695e-06,  5.8624e-06,  2.2539e-06, -1.8497e-05,  1.5404e-05,
          1.2181e-05, -1.4582e-05,  3.7801e-06,  1.0000e+00, -3.1552e-06],
        [ 9.1775e-06, -7.8198e-07,  2.2119e+00,  1.3297e-05,  1.0000e+00,
         -6.4646e-06,  5.3802e-06,  